# Image Feature Extraction **[DEMO]**

Determine which device to run PyTorch on:
- CUDA if an Nvidia GPU is installed
- CPU otherwise

In [1]:
import torch

# Set device to CUDA if we have an Nvidia GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device {device}")

Using device cuda


---
## Example 1

Load image datasets

In [2]:
from PIL import Image
import requests

img_urls = [
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/cats.png",
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/cats.jpeg",
]
image_real = Image.open(requests.get(img_urls[0], stream=True).raw).convert("RGB")
image_gen = Image.open(requests.get(img_urls[1], stream=True).raw).convert("RGB")

In [7]:
real_cow = Image.open(r"demo_imgs/Cow1.jpg")
gen_cow = Image.open(r"demo_imgs/Cow2.jpg")

Import pretrained image processor and ML model

In [8]:
from transformers import AutoImageProcessor, AutoModel

processor = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224")
model = AutoModel.from_pretrained("google/vit-base-patch16-224").to(device)

/home/bdsc/.local/share/virtualenvs/Computer-Vision-Pipeline-Gk3xQTeE/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 198/198 [00:00<00:00, 16966.76it/s]
[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Basic inference function

In [9]:
def infer(image):
    inputs = processor(image, return_tensors="pt").to(device)
    outputs = model(**inputs)
    return outputs.pooler_output

Pass images to inference function to obtain embeddings

In [10]:
embed_real = infer(image_real)
embed_gen = infer(image_gen)

In [11]:
embed_real_cow = infer(real_cow)
embed_gen_cow = infer(gen_cow)

Calculate similarity scores

In [12]:
from torch.nn.functional import cosine_similarity

similarity_score = cosine_similarity(embed_real, embed_gen, dim=1)
print(similarity_score)

tensor([0.5912], device='cuda:0', grad_fn=<SumBackward1>)


In [13]:
cow_sim_score = cosine_similarity(embed_real_cow, embed_gen_cow, dim=1)
print(cow_sim_score)

tensor([0.6718], device='cuda:0', grad_fn=<SumBackward1>)
